# Agentic GIS — programmatic quickstart

Run the same LangGraph agent the Streamlit app uses, from Python. Works offline in **demo** mode with the
bundled sample data (synthetic complaints — results are not findings about NYC).

Start Jupyter from the repository root, e.g. `uv run --with jupyter jupyter lab`.

In [ ]:
from agentic_gis.agent.graph import AgentSession, create_runtime

# provider="demo" uses the deterministic planner; use "ollama" with a running Ollama server.
runtime = create_runtime(mode="demo", llm_provider="demo")
session = AgentSession(runtime)
print(runtime.llm_message)

In [ ]:
result = session.ask("Find locations in NYC with repeated illegal dumping complaints.")
print(result.status, result.plan_source)
print(result.answer["headline"])
for step in result.steps:
    print(step["step_id"], step["tool"], step["status"], step.get("summary"))

Follow-up questions reuse the conversation context:

In [ ]:
followup = session.ask("Which of those are near industrial properties?")
print(followup.answer["headline"])

Results are GeoDataFrames in the session's artifact store:

In [ ]:
art = runtime.store.get(followup.primary_artifact)
art.data.head()

In [ ]:
# Render the map layers produced by the agent (Leaflet via Folium)
from agentic_gis.ui.layers import MapLayerSpec
from agentic_gis.ui.maps import render_folium

specs = [MapLayerSpec.model_validate(layer) for layer in followup.layers]
frames = {s.id: runtime.store.get(s.artifact_id).data for s in specs}
# local_assets=False: in Jupyter, load Leaflet from its CDN instead of Streamlit's static server
render_folium(specs, frames, "CARTO Positron", synthetic=True, local_assets=False)

### SQL approval

With `AGIS_SQL_APPROVAL=always` (or any Databricks query), `ask()` returns `status == "awaiting_approval"`
and `result.pending_approval["queries"]` holds the exact SQL. Continue with `session.resume(True)` or
`session.resume(False)`.